In [ ]:
!unzip /content/images_cropped.zip

Archive:  /content/images_cropped.zip
   creating: images_cropped/
   creating: images_cropped/augmented/
   creating: images_cropped/augmented/neg/
  inflating: images_cropped/augmented/neg/TC001_39_0_1470.jpg  
  inflating: images_cropped/augmented/neg/TC001_39_0_5015.jpg  
  inflating: images_cropped/augmented/neg/TC001_39_0_5993.jpg  
  inflating: images_cropped/augmented/neg/TC001_39_0_7582.jpg  
  inflating: images_cropped/augmented/neg/TC001_39_0_7620.jpg  
  inflating: images_cropped/augmented/neg/TC002_39_0_1866.jpg  
  inflating: images_cropped/augmented/neg/TC002_39_0_6222.jpg  
  inflating: images_cropped/augmented/neg/TC002_39_0_7033.jpg  
  inflating: images_cropped/augmented/neg/TC002_39_0_8737.jpg  
  inflating: images_cropped/augmented/neg/TC002_39_0_9183.jpg  
  inflating: images_cropped/augmented/neg/TC003_37_0_1643.jpg  
  inflating: images_cropped/augmented/neg/TC003_37_0_2823.jpg  
  inflating: images_cropped/augmented/neg/TC003_37_0_5267.jpg  
  inflating: images

In [17]:
from keras.preprocessing.image import load_img, img_to_array
from keras.applications.vgg16 import preprocess_input
from keras.applications.vgg16 import VGG16

import numpy as np
import glob, os, re

In [18]:
inputDir = '/content/images_cropped/**/*.jpg'
outfile = 'ASD_vgg16.npz'

In [19]:
print("Extracting features...")
model = VGG16(weights="imagenet", include_top=False, pooling='avg')

features_list = []
labels_list = []
ids_list = []

Extracting features...


In [20]:
def extract_participant_id(path):
    base = os.path.basename(path)
    stem, _ = os.path.splitext(base)      # works even if there’s no extension
    parts = stem.split('_')
    if len(parts) < 2:
        raise ValueError("No underscore found in filename: " + base)
    # Prefer the first fully-numeric token after the first underscore
    for token in parts[1:]:
        if token.isdigit():
            return token                   # e.g., "TC176_30_0_1812" -> "30"
    # Fallback: digits at start of the first token after underscore
    m = re.match(r'^(\d+)', parts[1])
    if m:
        return m.group(1)
    raise ValueError("Could not parse participant ID from: " + base)

In [ ]:
for imgFile in glob.iglob(inputDir, recursive=True):
    print(imgFile)
    image = load_img(imgFile)
    image = img_to_array(image)
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    image = preprocess_input(image)

    features = model.predict(image)
    features_list.append(features.flatten())

    # Label detection
    if "neg" in imgFile.lower():
        label = 0
    elif "pos" in imgFile.lower():
        label = 1
    else:
        raise ValueError("Wrong label in filename: " + imgFile)
    labels_list.append(label)

    # Participant ID: digits after first underscore
    participantID=extract_participant_id(imgFile)
    #print(participantID)
    ids_list.append(participantID)

Streaming output truncated to the last 5000 lines.
/content/images_cropped/augmented/neg/TC176_30_0_1812.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
/content/images_cropped/augmented/neg/TC351_58_0_5397.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
/content/images_cropped/augmented/neg/TC047_37_0_2638.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
/content/images_cropped/augmented/neg/TC101_36_0_2928.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
/content/images_cropped/augmented/neg/TC111_37_0_7846.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
/content/images_cropped/augmented/neg/TC159_36_0_5531.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
/content/images_cropped/augmented/neg/TC209_47_0_3398.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
/content/images_cropped/augmented/neg/TC054_30_0_1802.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
/content/images_cropped/augmented/neg/TC144_37_0_1791.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
/content/images_cropped/augmented/neg/TC274_56_0_4218.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
# Convert to arrays
X = np.array(features_list, dtype='float32')
y = np.array(labels_list, dtype='int32')
IDs = np.array(ids_list, dtype='<U10')

print("Final shapes -> X:", X.shape, "y:", y.shape, "IDs:", IDs.shape)

# Save to NPZ
np.savez_compressed(outfile, X=X, y=y, IDs=IDs)
print("Saved:", outfile)
